# IFRS S1/S2 Automated Requirements Extraction Notebook

This notebook builds a practical requirements knowledge base from the source IFRS S1 and IFRS S2 PDFs.

Design principle: **do not hardcode IFRS paragraph ranges**. The notebook automatically detects the official table of contents, extracts paragraph IDs from the PDF layout, derives section ranges from the contents, maps appendix guidance by referenced paragraphs, splits paragraphs into requirement clauses, tags evidence needs, validates the result, and exports JSON/JSONL/CSV outputs.

The only human intent kept as configuration is:

- which PDFs are the source files;
- which five report sections are in scope;
- which appendices are excluded because they are definitions or transition/legal context rather than report-generation requirements.


## 1. What is automatic here?

The notebook automatically performs these steps:

1. Reads IFRS S1 and IFRS S2 PDFs using PDF layout.
2. Detects the official contents page and extracts section starts such as `Governance 26`, `Strategy 28`, `Metrics and targets 45`, etc.
3. Computes paragraph ranges from the detected contents entries.
4. Extracts all numbered paragraphs, including appendix paragraphs such as `B62A`.
5. Maps body paragraphs to the five report sections using the detected contents structure.
6. Maps appendix guidance to the correct section by reading referenced paragraph numbers in headings/text.
7. Splits paragraphs into clause-level requirements where possible.
8. Tags requirements for validation and evidence mapping.
9. Exports a structured knowledge base for later report-generation agents.

This is a much stronger engineering design than manually writing `S1 Strategy = 28–42` in the notebook.


In [1]:
from pathlib import Path
import re, json
import fitz
import pandas as pd
from collections import defaultdict, Counter

PDF_SOURCES = {
    'IFRS S1': Path('gen_data/IFRS/ifrs_s1.pdf'),
    'IFRS S2': Path('gen_data/IFRS/ifrs_s2.pdf'),
}
TARGET_REPORT_SECTIONS = [
    'General Requirements','Governance','Strategy','Risk Management','Metrics and Targets'
]

PARA_ID_RE = re.compile(r'^(?P<id>[A-Z]?\d+[A-Z]?)(?:\s+|$)(?P<rest>.*)$')
LIST_MARKER_RE = re.compile(r'(?<!\w)(\([a-z]\)|\([ivxlcdm]+\)|\(\d+\))\s+', re.I)


def normalize_ws(text):
    if text is None: return ''
    text = text.replace('\u00ad','')
    text = text.replace('\ufffe','')
    text = re.sub(r'\s+', ' ', text)
    text = text.replace(' .', '.').replace(' ,', ',')
    return text.strip()


def paragraph_sort_key(pid):
    m = re.match(r'^([A-Z]?)(\d+)([A-Z]?)$', str(pid))
    if not m:
        return (99, 10**9, '')
    prefix, num, suffix = m.groups()
    prefix_rank = {'':0, 'B':1, 'C':2, 'D':3, 'E':4}.get(prefix, 50)
    suffix_rank = ord(suffix)-64 if suffix else 0
    return (prefix_rank, int(num), suffix_rank)


def paragraph_family(pid):
    m = re.match(r'^([A-Z]?)(\d+)([A-Z]?)$', str(pid))
    if not m: return ('', None, '')
    return m.group(1), int(m.group(2)), m.group(3)


def group_page_lines(page, y_tol=2.0):
    """Return visually ordered text lines using PyMuPDF word boxes.

    Using `get_text("words")` preserves spaces better than joining raw spans,
    while y-grouping lets us reunite margin paragraph IDs with the text line.
    """
    words = []
    for w in page.get_text('words'):
        x0, y0, x1, y1, text, *_ = w
        text = str(text).strip()
        if not text:
            continue
        words.append({'text': text, 'x0': x0, 'x1': x1, 'y0': y0, 'y1': y1})
    words = sorted(words, key=lambda t: (t['y0'], t['x0']))
    groups = []
    for word in words:
        if not groups or abs(groups[-1]['y'] - word['y0']) > y_tol:
            groups.append({'y': word['y0'], 'words': [word]})
        else:
            groups[-1]['words'].append(word)
            groups[-1]['y'] = sum(t['y0'] for t in groups[-1]['words']) / len(groups[-1]['words'])
    lines = []
    for g in groups:
        toks = sorted(g['words'], key=lambda t: t['x0'])
        text = normalize_ws(' '.join(t['text'] for t in toks))
        if text:
            lines.append({
                'text': text,
                'x0': min(t['x0'] for t in toks),
                'x1': max(t['x1'] for t in toks),
                'y': g['y'],
            })
    return lines


def extract_toc_entries(pdf_path, standard):
    doc = fitz.open(pdf_path)
    entries = []
    found_contents = False
    for page_index in range(min(8, len(doc))):
        lines = group_page_lines(doc[page_index])
        page_text = ' '.join(l['text'] for l in lines).upper().replace(' ', '')
        if 'CONTENTS' not in page_text and not found_contents:
            continue
        found_contents = True
        for line in lines:
            text = line['text']
            m = re.match(r'^(?P<title>.+?)\s+(?P<start>[A-Z]?\d+[A-Z]?)$', text)
            if not m:
                continue
            title = normalize_ws(m.group('title'))
            start = m.group('start')
            # drop page footer and non-standard titles
            if title.upper().startswith(('IFRS ', 'S1', 'S2', 'APPROVAL', 'FOR ', 'ILLUSTRATIVE', 'BASIS')):
                continue
            if title in {'© IFRS Foundation'}:
                continue
            entries.append({
                'standard': standard,
                'toc_title': title,
                'start_paragraph': start,
                'toc_page': page_index+1,
                'is_uppercase_heading': title.upper() == title,
            })
        # stop after contents page if next page does not look like continuation
        if found_contents and page_index > 0:
            # these PDFs have short TOCs; one page is enough
            break
    # compute end paragraph within the numeric body only from TOC entries
    body_entries = [e for e in entries if paragraph_family(e['start_paragraph'])[0] == '']
    # keep original order from contents
    for i, e in enumerate(body_entries):
        curr = paragraph_family(e['start_paragraph'])[1]
        next_start = None
        for nxt in body_entries[i+1:]:
            n = paragraph_family(nxt['start_paragraph'])[1]
            if n and n > curr:
                next_start = n
                break
        e['end_paragraph'] = (next_start - 1) if next_start else None
    # merge end back
    by_key = {(e['toc_title'], e['start_paragraph']): e for e in body_entries}
    for e in entries:
        if (e['toc_title'], e['start_paragraph']) in by_key:
            e['end_paragraph'] = by_key[(e['toc_title'], e['start_paragraph'])]['end_paragraph']
        else:
            e['end_paragraph'] = None
    return entries


def is_probable_heading(line_text, known_toc_titles=None):
    t = normalize_ws(line_text)
    if not t: return False
    if known_toc_titles and t.lower() in {h.lower() for h in known_toc_titles}:
        return True
    if re.match(r'^Appendix [A-Z]$', t):
        return True
    if len(t) <= 100 and '(' in t and re.search(r'paragraphs?\s+[A-Z]?\d+', t, re.I):
        return True
    if len(t) <= 90:
        # short lines with no sentence punctuation that are not list items are probably headings
        if not re.match(r'^\([a-zivxlcdm0-9]+\)', t, re.I) and not re.search(r'[.;:]$', t):
            words = re.findall(r'[A-Za-z]+', t)
            if words:
                uppercase = t.upper() == t
                titleish = sum(1 for w in words if w[:1].isupper()) >= max(1, len(words)//2)
                if uppercase or titleish:
                    return True
    return False


def extract_paragraphs(pdf_path, standard, known_toc_titles=None):
    doc = fitz.open(pdf_path)
    rows = []
    current = None
    current_heading = None
    current_appendix = None
    for page_index, page in enumerate(doc):
        for line in group_page_lines(page):
            txt = normalize_ws(line['text'])
            # crop repetitive headers/footers and footnotes
            if line['y'] < 108 or line['y'] > 690:
                continue
            if txt.startswith('© IFRS Foundation') or txt.startswith('IFRS SUSTAINABILITY'):
                continue
            if re.match(r'^\d+\s+© IFRS Foundation$', txt):
                continue
            # appendix heading detection before paragraph id check
            if re.match(r'^Appendix\s+[A-Z]$', txt):
                if current:
                    current['text'] = normalize_ws(' '.join(current['parts']))
                    del current['parts']
                    rows.append(current)
                    current = None
                current_appendix = txt.split()[-1]
                current_heading = txt
                continue
            m = PARA_ID_RE.match(txt)
            is_pid = False
            pid = None
            rest = ''
            if m:
                cand = m.group('id')
                prefix, num, suffix = paragraph_family(cand)
                # paragraph ID appears near text column/margin; avoid regular sentence numbers in body text
                if num is not None and line['x0'] <= 155 and not (prefix == '' and num < 10 and line['y'] > 660):
                    # skip contents page entries like OBJECTIVE 1 by requiring candidate at beginning and line after page 4? not an issue because we exclude first pages by no current body? Actually toc y in pages may pass.
                    # avoid parsing table of contents by skipping pages before actual standard begins unless line text starts with paragraph id and has prose.
                    if not (page_index < 5 and standard == 'IFRS S1') and not (page_index < 5 and standard == 'IFRS S2'):
                        is_pid = True
                        pid = cand
                        rest = normalize_ws(m.group('rest'))
            if is_pid:
                if current:
                    current['text'] = normalize_ws(' '.join(current['parts']))
                    del current['parts']
                    rows.append(current)
                current = {
                    'standard': standard,
                    'paragraph_id': pid,
                    'page': page_index + 1,
                    'appendix': current_appendix,
                    'nearest_heading': current_heading,
                    'parts': [rest] if rest else []
                }
                continue
            # heading-like lines should update context, not contaminate prior paragraph text
            if is_probable_heading(txt, known_toc_titles):
                current_heading = txt
                continue
            if current:
                current['parts'].append(txt)
    if current:
        current['text'] = normalize_ws(' '.join(current['parts']))
        del current['parts']
        rows.append(current)
    return rows


def classify_toc_title(title, standard):
    t = title.lower().strip()
    if t == 'governance': return 'Governance'
    if t == 'strategy': return 'Strategy'
    if t == 'risk management': return 'Risk Management'
    if t == 'metrics and targets': return 'Metrics and Targets'
    # Everything outside the four core-content headings that is still a disclosure requirement backbone is General Requirements.
    # This uses heading semantics, not paragraph ranges.
    general_keywords = [
        'objective','scope','conceptual','fair presentation','materiality','reporting entity',
        'connected information','core content','general requirements','sources of guidance',
        'location of disclosures','timing of reporting','comparative information','statement of compliance',
        'judgements','measurement uncertainty','errors'
    ]
    if any(k in t for k in general_keywords):
        return 'General Requirements'
    return None


def build_body_section_map(toc_entries, paragraphs_df):
    rows=[]
    # Prepare paragraph numeric set per standard
    existing_nums = {}
    body_only = paragraphs_df[paragraphs_df['paragraph_prefix'].eq('') & paragraphs_df['appendix'].isna()]
    for std, grp in body_only.groupby('standard'):
        existing_nums[std] = sorted(grp['paragraph_number'].dropna().astype(int).unique())
    for e in toc_entries:
        section = classify_toc_title(e['toc_title'], e['standard'])
        if not section: continue
        start = paragraph_family(e['start_paragraph'])[1]
        end = e.get('end_paragraph')
        if start is None: continue
        if end is None:
            nums = [n for n in existing_nums.get(e['standard'], []) if n >= start]
            end = max(nums) if nums else start
        # Avoid assigning duplicate parent headings with same start to General if a more specific target heading starts at same paragraph.
        rows.append({**e, 'report_section': section, 'start_number': start, 'end_number': end})
    # remove duplicate same start where target core-specific beats parent general
    priority = {'Governance':4,'Strategy':4,'Risk Management':4,'Metrics and Targets':4,'General Requirements':1}
    unique=[]
    by_std_start=defaultdict(list)
    for r in rows: by_std_start[(r['standard'],r['start_number'])].append(r)
    for key,lst in by_std_start.items():
        lst=sorted(lst,key=lambda x: priority[x['report_section']], reverse=True)
        # keep the best; if it is General only, keep it
        unique.append(lst[0])
    return unique


def assign_body_sections(paragraphs_df, section_ranges):
    df = paragraphs_df.copy()
    df['report_section'] = None
    df['official_section_heading'] = None
    df['mapping_method'] = None
    # sort specific ranges by length to assign smallest matched section first
    ranges = sorted(section_ranges, key=lambda r: (r['standard'], r['end_number']-r['start_number']))
    for idx, row in df.iterrows():
        if row['paragraph_prefix'] != '' or pd.notna(row.get('appendix', None)):
            continue
        num = row['paragraph_number']
        matches = [r for r in ranges if r['standard']==row['standard'] and r['start_number'] <= num <= r['end_number']]
        if matches:
            # choose most specific: shortest range, then core-specific priority
            matches = sorted(matches, key=lambda r: (r['end_number']-r['start_number'], 0 if r['report_section']!='General Requirements' else 1))
            m=matches[0]
            df.at[idx,'report_section'] = m['report_section']
            df.at[idx,'official_section_heading'] = m['toc_title']
            df.at[idx,'mapping_method'] = 'auto_toc_body_range'
    return df


def extract_referenced_paragraphs(text):
    # captures paragraph 22, paragraphs B1–B18, paragraph 29(a), see paragraphs 33–36
    refs=[]
    for m in re.finditer(r'paragraphs?\s+([A-Z]?\d+[A-Z]?)(?:\s*[–-]\s*([A-Z]?\d+[A-Z]?))?', text, re.I):
        start, end = m.group(1), m.group(2)
        refs.append(start)
        if end: refs.append(end)
    return refs


def map_appendix_sections(df):
    # for appendix paragraphs, infer section from their nearest heading's referenced paragraph, then paragraph text refs, then heading keywords
    body_map = { (r.standard, r.paragraph_id): r.report_section for r in df.itertuples() if r.paragraph_prefix=='' and pd.notna(r.report_section) }
    for idx,row in df[df['paragraph_prefix'].ne('')].iterrows():
        std = row['standard']
        candidates=[]
        for source_text in [row.get('nearest_heading',''), row.get('text','')]:
            candidates.extend(extract_referenced_paragraphs(str(source_text)))
        mapped=None
        for ref in candidates:
            # normalize if ref is appendix ref only; body refs matter most
            if (std, ref) in body_map:
                mapped=body_map[(std, ref)]; break
            # For ranges ending Bxx, use existing section of referenced B if already filled? not needed
        if not mapped:
            heading = str(row.get('nearest_heading','')).lower()
            para_text = str(row.get('text','')).lower()
            combined = heading + ' ' + para_text[:250]
            if any(k in combined for k in ['greenhouse gas','scope 3','financed emissions','commercial banking','asset management','insurance','gross exposure','carbon credit','climate-related targets','cross-industry metric','metrics']):
                mapped='Metrics and Targets'
            elif any(k in combined for k in ['climate resilience','scenario analysis']):
                mapped='Strategy'
            elif any(k in combined for k in ['risk management','identify, assess, prioritise']):
                mapped='Risk Management'
            elif any(k in combined for k in ['governance','board','management\'s role']):
                mapped='Governance'
            elif any(k in combined for k in ['materiality','connected information','sources of guidance','qualitative characteristics','reporting entity','cross-reference','comparative information','errors','commercially sensitive','law or regulation']):
                mapped='General Requirements'
        if mapped:
            df.at[idx,'report_section']=mapped
            df.at[idx,'official_section_heading']=row.get('nearest_heading')
            df.at[idx,'mapping_method']='auto_appendix_reference_or_heading'
    return df


def split_requirement_clauses(paragraph_text):
    text = normalize_ws(paragraph_text)
    matches=list(LIST_MARKER_RE.finditer(text))
    if len(matches) < 2:
        return [{'clause_marker': None, 'requirement_text': text, 'clause_level': 0}]
    preamble = text[:matches[0].start()].strip(' :;')
    clauses=[]
    for i,m in enumerate(matches):
        start=m.start(); end=matches[i+1].start() if i+1<len(matches) else len(text)
        marker=m.group(1)
        body=text[m.end():end].strip(' ;')
        marker_clean=marker.strip('()')
        level=1 if marker_clean.isalpha() and len(marker_clean)==1 else (2 if re.fullmatch(r'[ivxlcdm]+', marker_clean, re.I) else 3)
        req = normalize_ws((preamble + ' — ' if preamble else '') + marker + ' ' + body)
        clauses.append({'clause_marker': marker, 'requirement_text': req, 'clause_level': level})
    return clauses

KEYWORD_TAGS = {
    'governance_body': ['governance body','board','committee','individual(s) responsible'],
    'management_role': ['management’s role','management-level','management uses controls'],
    'strategy_decision_making': ['strategy and decision-making','responded to','plans to respond','transition plan'],
    'business_model_value_chain': ['business model','value chain'],
    'financial_effects': ['financial position','financial performance','cash flows','financial effects'],
    'scenario_analysis': ['scenario analysis','scenarios'],
    'risk_process': ['identify, assess, prioritise','risk management process','monitor climate-related risks','monitor sustainability-related risks'],
    'metrics': ['metrics','metric'],
    'targets': ['targets','target'],
    'ghg_emissions': ['greenhouse gas','ghg','co2 equivalent','scope 1','scope 2','scope 3'],
    'scope_1': ['scope 1'],
    'scope_2': ['scope 2'],
    'scope_3': ['scope 3'],
    'financed_emissions': ['financed emissions','category 15','asset management','commercial banking','insurance'],
    'commercial_banking': ['commercial banking','loans','project finance','undrawn loan commitments','gross exposure'],
    'carbon_credits': ['carbon credit','carbon credits'],
    'remuneration': ['remuneration'],
    'materiality': ['material','materiality'],
    'connected_information': ['connected information','connections between'],
    'source_guidance': ['sources of guidance','sasb','industry-based guidance'],
}

def infer_tags(text):
    tl=text.lower()
    return [tag for tag, kws in KEYWORD_TAGS.items() if any(k.lower() in tl for k in kws)]

def infer_obligation(text):
    tl=text.lower()
    if 'shall not' in tl or 'prohibited' in tl:
        return 'prohibition'
    if 'need not' in tl or 'permitted' in tl or 'may ' in tl or 'relief' in tl:
        return 'relief_or_optional'
    if 'shall' in tl or 'is required' in tl or 'requires an entity' in tl:
        return 'mandatory'
    return 'guidance'

def infer_banking_relevance(section, text, tags):
    if any(t in tags for t in ['financed_emissions','commercial_banking']): return 'high'
    if 'IFRS S2' in text and section in ['Strategy','Risk Management','Metrics and Targets']: return 'medium'
    if any(t in tags for t in ['financial_effects','scope_3','ghg_emissions']): return 'medium'
    return 'general'

def make_label(text):
    t = re.sub(r'^\([a-zivxlcdm0-9]+\)\s+', '', normalize_ws(text), flags=re.I)
    t = re.sub(r'^(An entity shall disclose|The entity shall disclose|Specifically, the entity shall disclose|To achieve this objective, an entity shall disclose)\s+', '', t, flags=re.I)
    words=t.split()
    return ' '.join(words[:14]) + ('…' if len(words)>14 else '')

def build_requirements_kb():
    toc_entries=[]
    for std,path in PDF_SOURCES.items():
        toc_entries.extend(extract_toc_entries(path,std))
    known_titles=[e['toc_title'] for e in toc_entries]
    paragraphs=[]
    for std,path in PDF_SOURCES.items():
        paragraphs.extend(extract_paragraphs(path,std,known_titles))
    df=pd.DataFrame(paragraphs)
    fam=df['paragraph_id'].apply(paragraph_family)
    df['paragraph_prefix']=[x[0] for x in fam]
    df['paragraph_number']=[x[1] for x in fam]
    df['paragraph_suffix']=[x[2] for x in fam]
    section_ranges=build_body_section_map(toc_entries, df)
    df=assign_body_sections(df, section_ranges)
    df=map_appendix_sections(df)
    # keep only selected sections
    selected=df[df['report_section'].isin(TARGET_REPORT_SECTIONS)].copy()
    # Exclude definition appendices and transition/amendment appendices; they are useful glossary/legal context, not report-generation requirements.
    selected = selected[~((selected['standard']=='IFRS S1') & (selected['appendix'].isin(['A','E'])))]
    selected = selected[~((selected['standard']=='IFRS S2') & (selected['appendix'].isin(['A','C'])))]
    selected=selected[selected['text'].str.len()>20].copy()
    req_rows=[]
    for row in selected.itertuples(index=False):
        for clause_index, clause in enumerate(split_requirement_clauses(row.text), start=1):
            req_text=clause['requirement_text']
            tags=infer_tags(req_text + ' ' + str(row.nearest_heading))
            obligation=infer_obligation(req_text)
            req_rows.append({
                'requirement_id': f"{row.standard.replace(' ','_')}_{row.paragraph_id}_C{clause_index:02d}",
                'standard': row.standard,
                'paragraph_id': row.paragraph_id,
                'page': row.page,
                'report_section': row.report_section,
                'official_section_heading': row.official_section_heading,
                'mapping_method': row.mapping_method,
                'clause_index': clause_index,
                'clause_marker': clause['clause_marker'],
                'clause_level': clause['clause_level'],
                'requirement_label': make_label(req_text),
                'requirement_text': req_text,
                'source_paragraph_text': row.text,
                'obligation_type': obligation,
                'mandatory': obligation in ['mandatory','prohibition'],
                'evidence_tags': tags,
                'banking_relevance': infer_banking_relevance(row.report_section, row.standard+' '+req_text, tags),
            })
    req_df=pd.DataFrame(req_rows)
    return toc_entries, df, selected, req_df, section_ranges

## 2. Run automatic extraction

This cell builds the full paragraph table, the selected paragraph table, and the final clause-level requirements knowledge base.


In [2]:
toc_entries, all_paragraphs_df, selected_paragraphs_df, requirements_kb_df, auto_section_ranges = build_requirements_kb()

print('Detected TOC entries:', len(toc_entries))
print('Extracted paragraphs:', len(all_paragraphs_df))
print('Selected paragraphs:', len(selected_paragraphs_df))
print('Requirement rows:', len(requirements_kb_df))

requirements_kb_df.groupby(['standard', 'report_section']).size().rename('requirement_count').reset_index()


Detected TOC entries: 31
Extracted paragraphs: 309
Selected paragraphs: 248
Requirement rows: 511


,standard,report_section,requirement_count
0,IFRS S1,General Requirements,165
1,IFRS S1,Governance,11
2,IFRS S1,Metrics and Targets,28
3,IFRS S1,Risk Management,10
4,IFRS S1,Strategy,34
5,IFRS S2,General Requirements,7
6,IFRS S2,Governance,11
7,IFRS S2,Metrics and Targets,151
8,IFRS S2,Risk Management,15
9,IFRS S2,Strategy,79


## 3. Inspect automatically detected section ranges

These ranges come from the PDFs' own contents page. They are not manually coded.


In [3]:
section_ranges_df = pd.DataFrame(auto_section_ranges)
section_ranges_df[['standard', 'toc_title', 'start_paragraph', 'end_paragraph', 'report_section', 'toc_page']]


,standard,toc_title,start_paragraph,end_paragraph,report_section,toc_page
0,IFRS S1,OBJECTIVE,1,4.0,General Requirements,4
1,IFRS S1,SCOPE,5,9.0,General Requirements,4
2,IFRS S1,CONCEPTUAL FOUNDATIONS,10,10.0,General Requirements,4
3,IFRS S1,Fair presentation,11,16.0,General Requirements,4
4,IFRS S1,Materiality,17,19.0,General Requirements,4
5,IFRS S1,Reporting entity,20,20.0,General Requirements,4
6,IFRS S1,Connected information,21,24.0,General Requirements,4
7,IFRS S1,CORE CONTENT,25,25.0,General Requirements,4
8,IFRS S1,Governance,26,27.0,Governance,4
9,IFRS S1,Strategy,28,42.0,Strategy,4


## 4. Validate the knowledge base

The checks below make the notebook safer for a regulatory extraction workflow. They verify traceability, duplicate IDs, allowed report sections, and banking-specific financed-emissions coverage.


In [4]:

def run_validation(requirements_df, selected_paragraphs_df):
    checks = []

    def add_check(name, passed, details):
        checks.append({'check': name, 'passed': bool(passed), 'details': details})

    allowed_sections = set(TARGET_REPORT_SECTIONS)
    actual_sections = set(requirements_df['report_section'].dropna().unique())
    add_check(
        'Only target report sections are present',
        actual_sections.issubset(allowed_sections),
        f'actual={sorted(actual_sections)}'
    )

    add_check(
        'Every requirement has source paragraph text',
        requirements_df['source_paragraph_text'].fillna('').str.len().gt(20).all(),
        f"empty_or_short={int(requirements_df['source_paragraph_text'].fillna('').str.len().le(20).sum())}"
    )

    add_check(
        'Requirement IDs are unique',
        requirements_df['requirement_id'].is_unique,
        f"duplicate_ids={int(requirements_df['requirement_id'].duplicated().sum())}"
    )

    add_check(
        'Every row has standard, paragraph, section, and page traceability',
        requirements_df[['standard','paragraph_id','report_section','page']].notna().all().all(),
        str(requirements_df[['standard','paragraph_id','report_section','page']].isna().sum().to_dict())
    )

    required_anchors = [
        ('IFRS S1','26'), ('IFRS S1','27'), ('IFRS S1','28'), ('IFRS S1','43'), ('IFRS S1','45'), ('IFRS S1','54'), ('IFRS S1','72'),
        ('IFRS S2','6'), ('IFRS S2','14'), ('IFRS S2','22'), ('IFRS S2','25'), ('IFRS S2','29'), ('IFRS S2','29A'), ('IFRS S2','29B'), ('IFRS S2','29C'),
        ('IFRS S2','33'), ('IFRS S2','36'), ('IFRS S2','B58'), ('IFRS S2','B59'), ('IFRS S2','B62'), ('IFRS S2','B62A')
    ]
    present = set(zip(selected_paragraphs_df['standard'], selected_paragraphs_df['paragraph_id']))
    missing = [x for x in required_anchors if x not in present]
    add_check('Required anchor paragraphs are captured', len(missing) == 0, f'missing={missing}')

    has_banking = requirements_df['evidence_tags'].apply(lambda tags: 'commercial_banking' in tags or 'financed_emissions' in tags).any()
    add_check('Banking / financed-emissions requirements are tagged', has_banking, 'expected tags: commercial_banking or financed_emissions')

    excluded_appendices_present = selected_paragraphs_df[
        ((selected_paragraphs_df['standard']=='IFRS S1') & selected_paragraphs_df['appendix'].isin(['A','E'])) |
        ((selected_paragraphs_df['standard']=='IFRS S2') & selected_paragraphs_df['appendix'].isin(['A','C']))
    ]
    add_check('Definition and transition appendices are excluded', excluded_appendices_present.empty, f'rows={len(excluded_appendices_present)}')

    return pd.DataFrame(checks)

validation_df = run_validation(requirements_kb_df, selected_paragraphs_df)
validation_df


,check,passed,details
0,Only target report sections are present,True,"actual=['General Requirements', 'Governance', ..."
1,Every requirement has source paragraph text,True,empty_or_short=0
2,Requirement IDs are unique,True,duplicate_ids=0
3,"Every row has standard, paragraph, section, an...",True,"{'standard': 0, 'paragraph_id': 0, 'report_sec..."
4,Required anchor paragraphs are captured,True,missing=[]
5,Banking / financed-emissions requirements are ...,True,expected tags: commercial_banking or financed_...
6,Definition and transition appendices are excluded,True,rows=0


## 5. Preview the final requirements knowledge base

Use this preview to check whether the output is useful for ESG report-generation agents.


In [5]:
preview_cols = [
    'requirement_id', 'standard', 'paragraph_id', 'page', 'report_section',
    'requirement_label', 'obligation_type', 'mandatory', 'banking_relevance', 'evidence_tags'
]
requirements_kb_df[preview_cols].head(20)


,requirement_id,standard,paragraph_id,page,report_section,requirement_label,obligation_type,mandatory,banking_relevance,evidence_tags
0,IFRS_S1_1_C01,IFRS S1,1,7,General Requirements,The objective of IFRS S1 General Requirements ...,guidance,False,general,[]
1,IFRS_S1_2_C01,IFRS S1,2,7,General Requirements,Information about sustainability-related risks...,guidance,False,medium,"[business_model_value_chain, financial_effects]"
2,IFRS_S1_3_C01,IFRS S1,3,7,General Requirements,This Standard requires an entity to disclose i...,mandatory,True,medium,[financial_effects]
3,IFRS_S1_4_C01,IFRS S1,4,7,General Requirements,This Standard also prescribes how an entity pr...,guidance,False,general,[]
4,IFRS_S1_5_C01,IFRS S1,5,7,General Requirements,An entity shall apply this Standard in prepari...,mandatory,True,general,[]
5,IFRS_S1_6_C01,IFRS S1,6,7,General Requirements,Sustainability-related risks and opportunities...,guidance,False,general,[]
6,IFRS_S1_7_C01,IFRS S1,7,7,General Requirements,Other IFRS Sustainability Disclosure Standards...,mandatory,True,general,[]
7,IFRS_S1_8_C01,IFRS S1,8,8,General Requirements,An entity may apply IFRS Sustainability Disclo...,relief_or_optional,False,general,[]
8,IFRS_S1_9_C01,IFRS S1,9,8,General Requirements,This Standard uses terminology suitable for pr...,guidance,False,general,[]
9,IFRS_S1_10_C01,IFRS S1,10,8,General Requirements,For sustainability-related financial informati...,guidance,False,general,[]


## 6. Banking and financed-emissions extraction check

This is important for banks because IFRS S2 requires additional information about financed emissions when activities include asset management, commercial banking, or insurance.


In [6]:
banking_rows = requirements_kb_df[requirements_kb_df['evidence_tags'].apply(
    lambda tags: ('financed_emissions' in tags) or ('commercial_banking' in tags)
)].copy()

banking_rows[[
    'requirement_id', 'standard', 'paragraph_id', 'page', 'report_section',
    'requirement_label', 'requirement_text', 'evidence_tags'
]].head(40)


,requirement_id,standard,paragraph_id,page,report_section,requirement_label,requirement_text,evidence_tags
179,IFRS_S1_B14_C02,IFRS S1,B14,29,General Requirements,The decisions of primary users relate to provi...,The decisions of primary users relate to provi...,"[commercial_banking, materiality]"
360,IFRS_S2_29_C17,IFRS S2,29,15,Metrics and Targets,information relevant to the cross-industry met...,An entity shall disclose information relevant ...,"[metrics, ghg_emissions, financed_emissions, c..."
371,IFRS_S2_29A_C01,IFRS S2,29A,17,Metrics and Targets,In preparing disclosures to meet the requireme...,In preparing disclosures to meet the requireme...,"[ghg_emissions, scope_3, financed_emissions, c..."
373,IFRS_S2_29B_C02,IFRS S2,29B,17,Metrics and Targets,If an entity applies the limitation in paragra...,If an entity applies the limitation in paragra...,"[ghg_emissions, scope_3, financed_emissions]"
374,IFRS_S2_29C_C01,IFRS S2,29C,17,Metrics and Targets,If an entity has included Category 15 greenhou...,If an entity has included Category 15 greenhou...,"[ghg_emissions, scope_3, financed_emissions]"
375,IFRS_S2_30_C01,IFRS S2,30,17,Metrics and Targets,In preparing disclosures to meet the requireme...,In preparing disclosures to meet the requireme...,[financed_emissions]
376,IFRS_S2_31_C01,IFRS S2,31,17,Metrics and Targets,In preparing disclosures to meet the requireme...,In preparing disclosures to meet the requireme...,[financed_emissions]
377,IFRS_S2_32_C01,IFRS S2,32,18,Metrics and Targets,industry-based metrics that are associated wit...,An entity shall disclose industry-based metric...,"[business_model_value_chain, metrics, financed..."
446,IFRS_S2_B37_C01,IFRS S2,B37,34,Metrics and Targets,An entity that participates in one or more fin...,An entity that participates in one or more fin...,"[ghg_emissions, scope_3, financed_emissions, c..."
465,IFRS_S2_B58_C01,IFRS S2,B58,38,Metrics and Targets,Entities participating in financial activities...,Entities participating in financial activities...,"[ghg_emissions, financed_emissions]"


## 7. Export outputs

The exported files are designed for downstream generation agents and validators.


In [7]:

OUTPUT_DIR = Path('gen_data/IFRS/ifrs_requirements_kb_outputs_auto')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# CSV-friendly version: convert list tags to pipe-separated text.
csv_df = requirements_kb_df.copy()
csv_df['evidence_tags'] = csv_df['evidence_tags'].apply(lambda x: '|'.join(x) if isinstance(x, list) else x)

json_path = OUTPUT_DIR / 'ifrs_s1_s2_requirements_kb_auto.json'
jsonl_path = OUTPUT_DIR / 'ifrs_s1_s2_requirements_kb_auto.jsonl'
csv_path = OUTPUT_DIR / 'ifrs_s1_s2_requirements_kb_auto.csv'
paragraphs_path = OUTPUT_DIR / 'ifrs_s1_s2_selected_paragraphs_auto.csv'
section_ranges_path = OUTPUT_DIR / 'ifrs_s1_s2_auto_section_ranges.csv'
validation_path = OUTPUT_DIR / 'ifrs_s1_s2_requirements_kb_auto_validation.csv'
audit_path = OUTPUT_DIR / 'ifrs_s1_s2_requirements_kb_auto_audit_summary.md'

records = requirements_kb_df.to_dict(orient='records')
json_path.write_text(json.dumps(records, indent=2, ensure_ascii=False), encoding='utf-8')
with jsonl_path.open('w', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

csv_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
selected_paragraphs_df.to_csv(paragraphs_path, index=False, encoding='utf-8-sig')
pd.DataFrame(auto_section_ranges).to_csv(section_ranges_path, index=False, encoding='utf-8-sig')
validation_df.to_csv(validation_path, index=False, encoding='utf-8-sig')

counts_md = requirements_kb_df.groupby(['standard','report_section']).size().reset_index(name='requirements').to_markdown(index=False)
validation_md = validation_df.to_markdown(index=False)
summary = f"""# IFRS S1/S2 Automated Requirements KB Audit Summary

## Extraction design

- Paragraph ranges were **not hardcoded**.
- Section starts were detected from each PDF's official contents page.
- Body section ranges were computed automatically from the detected starts.
- Appendix guidance was mapped by referenced paragraph numbers and heading evidence.
- Definition and transition appendices were excluded from the report-generation KB.

## Output counts

- Extracted paragraphs: {len(all_paragraphs_df)}
- Selected paragraphs: {len(selected_paragraphs_df)}
- Requirement rows: {len(requirements_kb_df)}

## Requirements by section

{counts_md}

## Validation checks

{validation_md}
"""
audit_path.write_text(summary, encoding='utf-8')

print('Exported files:')
for path in [json_path, jsonl_path, csv_path, paragraphs_path, section_ranges_path, validation_path, audit_path]:
    print('-', path)


Exported files:
- gen_data\IFRS\ifrs_requirements_kb_outputs_auto\ifrs_s1_s2_requirements_kb_auto.json
- gen_data\IFRS\ifrs_requirements_kb_outputs_auto\ifrs_s1_s2_requirements_kb_auto.jsonl
- gen_data\IFRS\ifrs_requirements_kb_outputs_auto\ifrs_s1_s2_requirements_kb_auto.csv
- gen_data\IFRS\ifrs_requirements_kb_outputs_auto\ifrs_s1_s2_selected_paragraphs_auto.csv
- gen_data\IFRS\ifrs_requirements_kb_outputs_auto\ifrs_s1_s2_auto_section_ranges.csv
- gen_data\IFRS\ifrs_requirements_kb_outputs_auto\ifrs_s1_s2_requirements_kb_auto_validation.csv
- gen_data\IFRS\ifrs_requirements_kb_outputs_auto\ifrs_s1_s2_requirements_kb_auto_audit_summary.md


## 8. How to use this KB downstream

For report-generation agents, use `report_section`, `requirement_text`, `paragraph_id`, `standard`, `evidence_tags`, and `banking_relevance`.

Recommended downstream pattern:

1. Select requirements for the report section being generated.
2. Filter to mandatory requirements first.
3. Use `evidence_tags` to retrieve bank ESG data, emissions data, financed-emissions data, risk data, governance evidence, and financial-impact evidence.
4. Ask the generation agent to cite `standard + paragraph_id` in its internal traceability metadata.
5. Run a validation agent that checks generated text against the same requirement rows.
